In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import VotingClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
import pandas as pd
from sklearn.model_selection import train_test_split

# Load your data
df = pd.read_csv("heart_failure_data.csv")
df['HF'] = df['HF'].apply(lambda x: 1 if x in [1, 2] else 0)

# Prepare different feature sets
X_all = df[['EF', 'GLS', 'QRS']]
X_no_gls = df[['EF', 'QRS']]
y = df['HF'].apply(lambda x: 1 if x in [1, 2] else 0)  # Binary classification

# Scale both feature sets
from sklearn.preprocessing import StandardScaler
scaler_all = StandardScaler()
X_all_scaled = scaler_all.fit_transform(X_all)

scaler_nogls = StandardScaler()
X_nogls_scaled = scaler_nogls.fit_transform(X_no_gls)

# Split data (same y)
from sklearn.model_selection import train_test_split
X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all_scaled, y, test_size=0.2, random_state=42, stratify=y
)

X_train_nogls, X_test_nogls, _, _ = train_test_split(
    X_nogls_scaled, y, test_size=0.2, random_state=42, stratify=y
)

# Define models using respective feature sets
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from xgboost import XGBClassifier

svm_model = SVC(kernel='rbf', C=10, probability=True, random_state=42)
rf_model = RandomForestClassifier(n_estimators=100, min_samples_split=10, random_state=42)
xgb_model = XGBClassifier(eval_metric='logloss', colsample_bytree=0.8, learning_rate=0.01,
                          max_depth=3, n_estimators=50, subsample=0.8, random_state=42)

# Fit models separately
svm_model.fit(X_train_all, y_train)
rf_model.fit(X_train_all, y_train)
xgb_model.fit(X_train_nogls, y_train)

# Custom VotingClassifier using pre-trained models with different inputs
class CustomVotingClassifier:
    def __init__(self, svm, rf, xgb):
        self.svm = svm
        self.rf = rf
        self.xgb = xgb

    def predict(self, X_all_test, X_nogls_test):
        import numpy as np
        svm_pred = self.svm.predict(X_all_test)
        rf_pred = self.rf.predict(X_all_test)
        xgb_pred = self.xgb.predict(X_nogls_test)
        # Majority voting
        predictions = np.array([svm_pred, rf_pred, xgb_pred])
        majority_vote = np.round(np.mean(predictions, axis=0)).astype(int)
        return majority_vote

# Instantiate and evaluate
voter = CustomVotingClassifier(svm_model, rf_model, xgb_model)
y_pred = voter.predict(X_test_all, X_test_nogls)

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
print("Voting Classifier Accuracy:", accuracy_score(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))



Voting Classifier Accuracy: 0.9583333333333334
Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.92      0.96        12
           1       0.92      1.00      0.96        12

    accuracy                           0.96        24
   macro avg       0.96      0.96      0.96        24
weighted avg       0.96      0.96      0.96        24

Confusion Matrix:
 [[11  1]
 [ 0 12]]
